In [1]:
#CE 311 K Project 
# import libraries
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

Environmental Engineering Calculations & Graphing
PFR + CSTR
Unit Converter
Maybe BOD?

In [13]:
from scipy.optimize import fsolve
import numpy as np

def cstr_calculator(Qins=None, Cins=None, Qouts=None, Cout=None, V=None, HRT=None):
    """
    Solves the steady-state CSTR mass balance for any single unknown.
        Flow : Σ(Qin_i) = Σ(Qout_j)
        Mass : Σ(Qin_i * Cin_i) = Cout * Σ(Qout_j)
        HRT  : HRT = V / Σ(Qout_j)
    Parameters:
        Qins  : list of floats  — inlet flow rates
        Cins  : list of floats  — inlet concentrations (same length as Qins)
        Qouts : list of floats  — outlet flow rates
        Cout  : float           — outlet concentration (shared by all outlets)
        V     : float           — reactor volume
        HRT   : float           — hydraulic retention time (V / ΣQout)
    """

    # --- V / HRT are linked via HRT = V / ΣQout ---
    # Defer computing whichever is missing until after Qouts is known
    v_unknown   = V   is None
    hrt_unknown = HRT is None

    if v_unknown and hrt_unknown:
        raise ValueError(
            "Provide at least one of V or HRT. "
            "V cannot be solved from mass balance alone (no reaction term). "
            "Supply HRT to back-calculate V."
        )

    unknowns = {
        "Qins":  Qins  is None,
        "Cins":  Cins  is None,
        "Qouts": Qouts is None,
        "Cout":  Cout  is None,
    }

    num_unknowns = sum(unknowns.values())
    if num_unknowns != 1:
        raise ValueError(f"Exactly 1 of (Qins, Cins, Qouts, Cout) must be None (got {num_unknowns}).")

    if Qins is not None and Cins is not None and len(Qins) != len(Cins):
        raise ValueError(f"Qins (len={len(Qins)}) and Cins (len={len(Cins)}) must be the same length.")

    # --- Solver ---
    def equation(x):
        _Qins  = x.tolist()  if unknowns["Qins"]  else Qins
        _Cins  = x.tolist()  if unknowns["Cins"]  else Cins
        _Qouts = x.tolist()  if unknowns["Qouts"] else Qouts
        _Cout  = float(x[0]) if unknowns["Cout"]  else Cout

        total_Qin  = sum(_Qins)
        total_Qout = sum(_Qouts)
        inlet_load = sum(q * c for q, c in zip(_Qins, _Cins))

        if unknowns["Qouts"]:
            return total_Qin - total_Qout           # flow balance
        return inlet_load - _Cout * total_Qout      # mass balance

    x0 = [1.0]
    solution, info, ier, msg = fsolve(equation, x0, full_output=True)

    if ier != 1:
        raise RuntimeError(f"Solver failed to converge: {msg}")

    solved = float(solution[0])

    # --- Reconstruct full variable set ---
    _Qins  = Qins  if not unknowns["Qins"]  else [solved]
    _Cins  = Cins  if not unknowns["Cins"]  else [solved]
    _Qouts = Qouts if not unknowns["Qouts"] else [solved]
    _Cout  = Cout  if not unknowns["Cout"]  else solved

    total_Qin  = sum(_Qins)
    total_Qout = sum(_Qouts)

    # --- Flow balance check ---
    if not unknowns["Qins"] and not unknowns["Qouts"]:
        if not np.isclose(total_Qin, total_Qout, rtol=0.01):
            raise ValueError(
                f"Flow imbalance: sum(Qins)={total_Qin} ≠ sum(Qouts)={total_Qout}. "
                "No steady-state solution exists."
            )

    # --- V / HRT resolution (now that Qouts is known) ---
    if not v_unknown and not hrt_unknown:
        pass  # both provided, use as-is
    elif v_unknown:
        V = HRT * total_Qout
    elif hrt_unknown:
        HRT = V / total_Qout

    return {
        "Qins":         _Qins,
        "Cins":         _Cins,
        "Qouts":        _Qouts,
        "Cout":         float(_Cout),
        "V":            float(V),
        "HRT":          float(HRT),
        "total_Qin":    float(total_Qin),
        "total_Qout":   float(total_Qout),
        "solved_for":   [k for k, v in unknowns.items() if v][0],
        "solved_value": float(solved),
    }


def cstr_sweep(sweep_param, sweep_values, **kwargs):
    """
    Runs cstr_calculator across a range of values for one parameter,
    returning results as numpy arrays ready for plotting.

    Parameters:
        sweep_param  : str   — name of the parameter to vary
                               ('Qins', 'Cins', 'Qouts', 'Cout', 'V', 'HRT')
        sweep_values : array — values to sweep over
        **kwargs     : all other cstr_calculator parameters (fixed)

    Returns:
        dict of numpy arrays, one value per sweep point:
    """

    list_params = {"Qins", "Cins", "Qouts"}

    records = {
        "sweep_values": [],
        "Cout":         [],
        "V":            [],
        "HRT":          [],
        "total_Qin":    [],
        "total_Qout":   [],
        "solved_values":[],
    }

    for val in sweep_values:
        params = dict(kwargs)

        # List params wrap the scalar in a list to match expected input type
        if sweep_param in list_params:
            params[sweep_param] = [val]
        else:
            params[sweep_param] = val

        result = cstr_calculator(**params)

        records["sweep_values"].append(val)
        records["Cout"].append(result["Cout"])
        records["V"].append(result["V"])
        records["HRT"].append(result["HRT"])
        records["total_Qin"].append(result["total_Qin"])
        records["total_Qout"].append(result["total_Qout"])
        records["solved_values"].append(result["solved_value"])

    return {k: np.array(v) for k, v in records.items()}

a = cstr_calculator(Qins=[10, 20], Cins=[100, 200], Qouts=[30], Cout=None, V=500)
print(a)

{'Qins': [10, 20], 'Cins': [100, 200], 'Qouts': [30], 'Cout': 166.66666666666666, 'V': 500.0, 'HRT': 16.666666666666668, 'total_Qin': 30.0, 'total_Qout': 30.0, 'solved_for': 'Cout', 'solved_value': 166.66666666666666}


In [ ]:
#cstr in series solver

#equation Cn=Co**4*(t/HRT)**n-1 *1/(n-1)! *np.e**-t/HRT


from scipy.optimize import fsolve
from scipy.integrate import solve_ivp
from scipy.special import factorial
import numpy as np

def cstr_series_calc(Co=None, total_HRT=None, n=None, volumes=None, Q=None, Cn=None, t=None):
    """
    Solves the tanks-in-series RTD model for CSTRs in series.

    Two modes depending on inputs:
        Equal volumes   : analytical formula using total_HRT and n
        Unequal volumes : numerical ODE simulation using volumes list + Q

    Analytical formula (equal volumes, pulse tracer input):
        HRT_i = total_HRT / n
        C(t) = Co * (1/HRT_i) * (t/HRT_i)^(n-1) / (n-1)! * exp(-t/HRT_i)

    Parameters:
        Co        : float       — inlet pulse concentration [mg/L]
        total_HRT : float       — total HRT across all tanks = Σ(V_i)/Q
        n         : int         — number of CSTRs in series
        volumes   : list        — individual tank volumes (unequal mode, requires Q)
        Q         : float       — flow rate (required for unequal volume mode)
        Cn        : float       — outlet concentration at time t to solve for/against
        t         : float or array — time(s) at which to evaluate C(t)
                                    if array, returns full C(t) curve (no unknown solving)

    Pass exactly ONE of (Co, total_HRT, Cn) as None to solve for it.
    n must always be provided.
    t must always be provided.

    Returns:
        dict:
            "Co"          : float
            "total_HRT"   : float
            "HRT_per_tank": list of floats  — one per tank
            "n"           : int
            "volumes"     : list of floats
            "Q"           : float or None
            "t"           : array
            "C_t"         : array  — concentration profile over time (plottable)
            "Cn"          : float  — concentration at the queried t
            "solved_for"  : str
            "solved_value": float
    """

    if n is None:
        raise ValueError("n (number of reactors) must always be provided and must be an integer.")
    n = int(n)

    if t is None:
        raise ValueError("t (time) must always be provided.")

    t_array = np.atleast_1d(np.asarray(t, dtype=float))
    unequal_mode = volumes is not None

    # --- Validate inputs for each mode ---
    if unequal_mode:
        if len(volumes) != n:
            raise ValueError(f"Length of volumes ({len(volumes)}) must equal n ({n}).")
        if Q is None:
            raise ValueError("Q (flow rate) is required when volumes are provided.")
        HRT_per_tank = [v / Q for v in volumes]
        _total_HRT   = sum(HRT_per_tank)
    else:
        if total_HRT is None and Cn is not None:
            raise ValueError("total_HRT required in equal-volume mode unless solving for it.")
        _total_HRT   = total_HRT
        HRT_per_tank = [total_HRT / n] * n if total_HRT is not None else None

    # --- Core concentration function ---
    def compute_C_at_t(Co_, total_HRT_, t_val):
        """Compute outlet concentration at a single time value."""
        if unequal_mode:
            # Numerical ODE: dC_i/dt = (C_{i-1} - C_i) / HRT_i
            # Initial conditions: C_0 = Co_ (pulse → spike at t=0), all tanks start at 0
            # We simulate with a narrow Gaussian pulse approximating delta(t)
            def odes(time, C):
                dC = np.zeros(n)
                for i in range(n):
                    C_in = Co_ if i == 0 else C[i - 1]
                    dC[i] = (C_in - C[i]) / HRT_per_tank[i]
                return dC

            sol = solve_ivp(
                odes,
                t_span=(0, max(t_val * 3, total_HRT_ * 5)),
                y0=np.zeros(n),
                t_eval=np.atleast_1d(t_val),
                method="RK45",
                dense_output=True,
            )
            return float(sol.y[-1, 0])  # last tank, at t_val

        else:
            # Analytical formula for equal volumes
            HRT_i = total_HRT_ / n
            if HRT_i <= 0 or t_val < 0:
                return 0.0
            return float(
                Co_ * (1 / HRT_i) * (t_val / HRT_i) ** (n - 1)
                / factorial(n - 1) * np.exp(-t_val / HRT_i)
            )

    def compute_C_profile(Co_, total_HRT_):
        """Compute full C(t) array over t_array for plotting."""
        if unequal_mode:
            def odes(time, C):
                dC = np.zeros(n)
                for i in range(n):
                    C_in = Co_ if i == 0 else C[i - 1]
                    dC[i] = (C_in - C[i]) / HRT_per_tank[i]
                return dC

            t_span = (0, max(t_array[-1], total_HRT_ * 5))
            sol = solve_ivp(
                odes,
                t_span=t_span,
                y0=np.zeros(n),
                t_eval=t_array,
                method="RK45",
                dense_output=True,
            )
            return sol.y[-1]  # last tank concentration over time

        else:
            HRT_i = total_HRT_ / n
            return (
                Co_ * (1 / HRT_i) * (t_array / HRT_i) ** (n - 1)
                / factorial(n - 1) * np.exp(-t_array / HRT_i)
            )

    # --- Identify unknown and solve ---
    unknowns = {
        "Co":        Co        is None,
        "total_HRT": total_HRT is None and not unequal_mode,
        "Cn":        Cn        is None,
    }

    num_unknowns = sum(unknowns.values())

    # If t is array and no unknown, just return the full profile
    if num_unknowns == 0 or (num_unknowns == 1 and unknowns["Cn"] and len(t_array) > 1):
        C_profile = compute_C_profile(Co, _total_HRT)
        return {
            "Co":           float(Co),
            "total_HRT":    float(_total_HRT),
            "HRT_per_tank": HRT_per_tank,
            "n":            n,
            "volumes":      volumes if volumes else [_total_HRT / n * (Q or 1)] * n,
            "Q":            Q,
            "t":            t_array,
            "C_t":          C_profile,
            "Cn":           float(C_profile[-1]),
            "solved_for":   "none",
            "solved_value": float(C_profile[-1]),
        }

    if num_unknowns != 1:
        raise ValueError(f"Exactly 1 of (Co, total_HRT, Cn) must be None (got {num_unknowns}).")

    t_solve = float(t_array[0])  # use first t value for single-point solving

    def equation(x):
        _Co        = float(x[0]) if unknowns["Co"]        else Co
        _total_HRT = float(x[0]) if unknowns["total_HRT"] else _total_HRT if not unequal_mode else sum(HRT_per_tank)
        _Cn        = Cn          if not unknowns["Cn"]     else compute_C_at_t(_Co, _total_HRT, t_solve)

        return compute_C_at_t(_Co, _total_HRT, t_solve) - _Cn

    x0 = [1.0]
    solution, info, ier, msg = fsolve(equation, x0, full_output=True)

    if ier != 1:
        raise RuntimeError(f"Solver failed to converge: {msg}")

    solved = float(solution[0])

    _Co        = Co        if not unknowns["Co"]        else solved
    _total_HRT = total_HRT if not unknowns["total_HRT"] else solved
    if unequal_mode:
        _total_HRT = sum(HRT_per_tank)
    _Cn        = Cn        if not unknowns["Cn"]        else solved

    C_profile    = compute_C_profile(_Co, _total_HRT)
    HRT_per_tank = HRT_per_tank if unequal_mode else [_total_HRT / n] * n

    return {
        "Co":           float(_Co),
        "total_HRT":    float(_total_HRT),
        "HRT_per_tank": HRT_per_tank,
        "n":            n,
        "volumes":      volumes if volumes else [_total_HRT / n * (Q or 1)] * n,
        "Q":            Q,
        "t":            t_array,
        "C_t":          C_profile,
        "Cn":           float(_Cn),
        "solved_for":   [k for k, v in unknowns.items() if v][0],
        "solved_value": float(solved),
    }


def cstr_series_sweep(sweep_param, sweep_values, **kwargs):
    """
    Sweeps one parameter and returns plottable arrays.

    Most useful sweeps:
        sweep_param="n"         → see how adding reactors sharpens the RTD
        sweep_param="total_HRT" → see how HRT shifts the peak

    Returns:
        dict of np.arrays, keyed by result field.
        "C_t" is a 2D array: shape (len(sweep_values), len(t))

    Example:
        t = np.linspace(0, 50, 500)
        results = cstr_series_sweep(
            sweep_param="n",
            sweep_values=[1, 2, 5, 10],
            Co=100, total_HRT=10, Cn=None, t=t
        )
        for i, n in enumerate([1, 2, 5, 10]):
            plt.plot(results["t"], results["C_t"][i], label=f"n={n}")
    """
    records = {k: [] for k in ["Co", "total_HRT", "n", "Cn", "t", "C_t", "sweep_values"]}

    for val in sweep_values:
        params = dict(kwargs)
        params[sweep_param] = val
        result = cstr_series_calc(**params)

        records["sweep_values"].append(val)
        records["Co"].append(result["Co"])
        records["total_HRT"].append(result["total_HRT"])
        records["n"].append(result["n"])
        records["Cn"].append(result["Cn"])
        records["t"].append(result["t"])
        records["C_t"].append(result["C_t"])

    return {
        "sweep_values": np.array(records["sweep_values"]),
        "Co":           np.array(records["Co"]),
        "total_HRT":    np.array(records["total_HRT"]),
        "n":            np.array(records["n"]),
        "Cn":           np.array(records["Cn"]),
        "t":            records["t"][0],          # shared time axis
        "C_t":          np.array(records["C_t"]), # shape: (n_sweep, n_t)
    }


In [ ]:
#new code

def f(x):
    return np.exp(-x**2)